## Exercises XP

In [1]:
 ### download and extract chinook sample DB
import urllib.request
import zipfile
from functools import partial
import os

chinook_url = 'http://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip'
if not os.path.exists('chinook.zip'):
    print('downloading chinook.zip ', end='')
    with urllib.request.urlopen(chinook_url) as response:
        with open('chinook.zip', 'wb') as f:
            for data in iter(partial(response.read, 4*1024), b''):
                print('.', end='', flush=True)
                f.write(data)

zipfile.ZipFile('chinook.zip').extractall()
assert os.path.exists('chinook.db')

downloading chinook.zip ...........................................................................

In [2]:
### useful: functions for displaying results from sql queries using pandas
from IPython.display import display
import pandas as pd

def sql(query):
    print()
    print(query)
    print()

def get_results(query):
    global engine
    q = query.statement if isinstance(query, sqlalchemy.orm.query.Query) else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    sql(query)

### Exercise 1 : Open the database

In [3]:
import sqlalchemy
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.orm import sessionmaker


In [4]:
# Crée un moteur pointant vers la base SQLite
engine = sqlalchemy.create_engine("sqlite:///chinook.db")

# Ouvre une connexion
cur = engine.connect()


In [5]:
# Crée un objet MetaData
metadata = sqlalchemy.MetaData()

# Réfléchit le schéma (charge toutes les tables existantes)
metadata.reflect(engine)


In [6]:
# Automap pour générer les classes à partir du schéma
Base = automap_base(metadata=metadata)
Base.prepare()

In [7]:
Session = sessionmaker(bind=engine)
session = Session()

###  Exercise 2 : table names

In [9]:
print(metadata.tables.keys())

dict_keys(['albums', 'artists', 'customers', 'employees', 'genres', 'invoice_items', 'tracks', 'media_types', 'invoices', 'playlist_track', 'playlists'])


### Exercise 3 : Tracks

In [26]:
# Track = Base.classes.tracks
# tracks = session.query(Track).limit(3).all()

# for t in tracks:
#     print(t.Name)

Track = Base.classes.tracks
query = session.query(Track).limit(3)
display_results(query)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,None,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99



SELECT tracks."TrackId" AS "tracks_TrackId", tracks."Name" AS "tracks_Name", tracks."AlbumId" AS "tracks_AlbumId", tracks."MediaTypeId" AS "tracks_MediaTypeId", tracks."GenreId" AS "tracks_GenreId", tracks."Composer" AS "tracks_Composer", tracks."Milliseconds" AS "tracks_Milliseconds", tracks."Bytes" AS "tracks_Bytes", tracks."UnitPrice" AS "tracks_UnitPrice" 
FROM tracks
 LIMIT ? OFFSET ?



### Exercise 4 : Albums from Tracks

In [28]:
# Album = Base.classes.albums
# Track = Base.classes.tracks

# results = session.query(Track.Name, Album.Title).join(Album).limit(20).all()

# for track_name, album_title in results:
#     print(f"{track_name} --> {album_title}")

Album = Base.classes.albums
Track = Base.classes.tracks

query = session.query(Track.Name, Album.Title).join(Album).limit(20)
display_results(query)


,Name,Title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Put The Finger On You,For Those About To Rock We Salute You
2,Let's Get It Up,For Those About To Rock We Salute You
3,Inject The Venom,For Those About To Rock We Salute You
4,Snowballed,For Those About To Rock We Salute You
5,Evil Walks,For Those About To Rock We Salute You
6,C.O.D.,For Those About To Rock We Salute You
7,Breaking The Rules,For Those About To Rock We Salute You
8,Night Of The Long Knives,For Those About To Rock We Salute You
9,Spellbound,For Those About To Rock We Salute You



SELECT tracks."Name" AS "tracks_Name", albums."Title" AS "albums_Title" 
FROM tracks JOIN albums ON albums."AlbumId" = tracks."AlbumId"
 LIMIT ? OFFSET ?



### Exercise 5 : Tracks sold

In [38]:
# InvoiceLine = Base.classes.invoice_items
# Track = Base.classes.tracks

# sales = session.query(InvoiceLine, Track.Name).join(Track).limit(10).all()

# for sale, track_name in sales:
#     print(f"Track: {track_name}, Quantity: {sale.Quantity}")

InvoiceLine = Base.classes.invoice_items
Track = Base.classes.tracks

query = session.query(Track.Name, InvoiceLine.Quantity).join(InvoiceLine).limit(10)
display_results(query)

,Name,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1



SELECT tracks."Name" AS "tracks_Name", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId"
 LIMIT ? OFFSET ?



### Exercise 6 : Top tracks sold

In [40]:
from sqlalchemy import func

Track = Base.classes.tracks
InvoiceLine = Base.classes.invoice_items

top_tracks = (
    session.query(Track.Name, func.sum(InvoiceLine.Quantity).label("total_sold"))
    .join(InvoiceLine)
    .group_by(Track.TrackId)
    .order_by(func.sum(InvoiceLine.Quantity).desc())
    .limit(10)
)

display_results(top_tracks)

,Name,total_sold
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
5,Not The Doctor,2
6,Por Causa De Você,2
7,Welcome Home (Sanitarium),2
8,Snowblind,2
9,Cornucopia,2



SELECT tracks."Name" AS "tracks_Name", sum(invoice_items."Quantity") AS total_sold 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId" GROUP BY tracks."TrackId" ORDER BY sum(invoice_items."Quantity") DESC
 LIMIT ? OFFSET ?



### Exercise 7 : Top selling artists

In [41]:
Artist = Base.classes.artists
Album = Base.classes.albums
Track = Base.classes.tracks
InvoiceLine = Base.classes.invoice_items

top_artists = (
    session.query(Artist.Name, func.sum(InvoiceLine.Quantity).label("total_sold"))
    .join(Album, Artist.ArtistId == Album.ArtistId)
    .join(Track, Album.AlbumId == Track.AlbumId)
    .join(InvoiceLine, Track.TrackId == InvoiceLine.TrackId)
    .group_by(Artist.ArtistId)
    .order_by(func.sum(InvoiceLine.Quantity).desc())
    .limit(10)
)

display_results(query)

,Name,Quantity
0,Balls to the Wall,1
1,Restless and Wild,1
2,Put The Finger On You,1
3,Inject The Venom,1
4,Evil Walks,1
5,Breaking The Rules,1
6,Dog Eat Dog,1
7,Overdose,1
8,Love In An Elevator,1
9,Janie's Got A Gun,1



SELECT tracks."Name" AS "tracks_Name", invoice_items."Quantity" AS "invoice_items_Quantity" 
FROM tracks JOIN invoice_items ON tracks."TrackId" = invoice_items."TrackId"
 LIMIT ? OFFSET ?

